# Fase 0 — Fundação

Objetivo desta Fase: **fechar o ciclo completo ponta-a-ponta** com uma trajetória real, por mais curta que seja.

> construir estrutura → simular → gravar DCD → abrir no MDTraj → plotar φ/ψ

Se este notebook rodar do início ao fim e produzir o mapa de Ramachandran no final, toda a infraestrutura do projeto está validada e as Fases 1–3 são só análise.

**Pré-requisito:** ambiente `md-ml` criado a partir de `environment.yml` (o Python 3.14 do sistema não serve).

```bash
conda env create -f environment.yml
conda activate md-ml
```

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import numpy as np
import matplotlib.pyplot as plt
import mdtraj as md
import openmm as mm

from src import config as cfg
from src.build_peptide import build_ala_peptide, write_pdb
from src.simulate import available_platforms, run

print("repositório :", REPO)
print("OpenMM      :", mm.version.version)
print("MDTraj      :", md.version.version)
print("dados em    :", cfg.DATA_ROOT)
cfg.ensure_dirs()

## 1. Validação da instalação

`CUDA` **precisa** aparecer na lista. Se não aparecer, todo o orçamento computacional do plano muda — resolver antes de seguir.

O equivalente pela linha de comando é `python -m openmm.testInstallation`, que além de listar as plataformas confere se elas concordam na energia (erro esperado da ordem de 1e-6).

In [ ]:
plats = available_platforms()
print("plataformas:", ", ".join(plats))

if "CUDA" in plats:
    print("\nOK — a RTX 4060 Ti será usada.")
else:
    print("\nATENÇÃO: sem CUDA. Verifique se o driver NVIDIA e o pacote")
    print("cuda-version do environment.yml estão instalados.")
    print("Fallback: --platform CPU (Sistema A ainda roda, mais devagar).")

## 2. Estruturas iniciais

Construídas por coordenadas internas em `src/build_peptide.py` — sem download, sem `tleap`, sem CHARMM-GUI. A geometria é verificada por `pytest` (comprimentos de ligação, reprodução de φ/ψ, ω *trans*, quiralidade **L** e parâmetros da α-hélice).

- **Sistema A** — ACE-ALA-NME, 22 átomos, iniciado na região α_R
- **Sistema B** — ACE-(ALA)₁₀-NME, 112 átomos, iniciado em α-hélice ideal

In [ ]:
write_pdb(build_ala_peptide(1, -60.0, -45.0),
          cfg.STRUCTURES / "alanine_dipeptide.pdb", "Sistema A: ACE-ALA-NME")
write_pdb(build_ala_peptide(10, -57.0, -47.0),
          cfg.STRUCTURES / "ala10.pdb", "Sistema B: ACE-(ALA)10-NME")

for spec in cfg.SYSTEMS.values():
    t = md.load(str(spec.pdb))
    print(f"Sistema {spec.key}: {t.n_atoms:3d} átomos, "
          f"{t.n_residues} resíduos — {spec.pdb.name}")
    print(f"            {spec.label}")

In [ ]:
# Inspeção visual (opcional). Confirme que o Sistema B saiu como uma hélice
# DESTRA — hélice canhota indica inversão de sinal na construção.
import nglview as nv

view = nv.show_mdtraj(md.load(str(cfg.SYSTEMS["B"].pdb)))
view.add_ball_and_stick()
view

## 3. Teste de fumaça: 1 ns do Sistema A

Exercita exatamente o mesmo código das réplicas de produção — só que curto. Solvatação em TIP3P, minimização, 20 ps de equilibração e 1 ns de produção gravando a cada 2 ps.

Na 4060 Ti isso leva poucos minutos. **Anote a velocidade em ns/dia** que aparece no log: ela calibra a estimativa de duração das réplicas de 500 ns.

In [ ]:
smoke_dir = cfg.RAW / "sysA" / "smoke"

traj_path = run(
    system="A",
    replica=0,
    ns=1.0,
    equil_ns=0.02,
    report_ps=2.0,
    out_dir=smoke_dir,
)

In [ ]:
import json
meta = json.loads((smoke_dir / "metadata.json").read_text(encoding="utf-8"))
for k in ("plataforma", "n_atomos_total", "n_atomos_soluto", "n_frames",
          "duracao_s", "ns_por_dia"):
    print(f"{k:18s}: {meta[k]}")

print(f"\nEstimativa para 3 x 500 ns: "
      f"{3 * 500 / meta['ns_por_dia']:.1f} dias de máquina")

## 4. Leitura da trajetória e extração de φ/ψ

A trajetória contém apenas os 22 átomos do soluto — a água foi descartada na gravação (nenhum descritor do projeto a usa, e mantê-la multiplicaria o arquivo por ~200).

In [ ]:
traj = md.load(str(traj_path), top=str(smoke_dir / "topology.pdb"))
print(traj)

_, phi = md.compute_phi(traj)
_, psi = md.compute_psi(traj)
phi_deg = np.degrees(phi[:, 0])
psi_deg = np.degrees(psi[:, 0])

print(f"\n{traj.n_frames} frames, φ e ψ extraídos")
print(f"φ: {phi_deg.min():7.1f} .. {phi_deg.max():7.1f} graus")
print(f"ψ: {psi_deg.min():7.1f} .. {psi_deg.max():7.1f} graus")

## 5. Mapa de Ramachandran

**Este é o artefato-entregável da Fase 0.**

Com apenas 1 ns não se espera ver todas as bacias metaestáveis — o dipeptídeo passa a maior parte do tempo na região onde começou. O que se espera ver é a densidade concentrada em φ < 0, na região α_R/C7eq. Uma nuvem em φ > 0 indicaria que a estrutura inicial saiu espelhada.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(phi_deg, psi_deg, s=6, alpha=0.35, c=traj.time, cmap="viridis")
ax1.set(xlim=(-180, 180), ylim=(-180, 180),
        xticks=range(-180, 181, 90), yticks=range(-180, 181, 90),
        xlabel=r"$\phi$ (graus)", ylabel=r"$\psi$ (graus)",
        title=f"Ramachandran — {traj.n_frames} frames (1 ns)")
ax1.axhline(0, lw=0.5, c="gray"); ax1.axvline(0, lw=0.5, c="gray")
ax1.set_aspect("equal")

# Bacias de referência da literatura, para conferência visual.
for nome, (x, y) in {r"$\alpha_R$": (-63, -43), "C7eq": (-82, 73),
                     "C5": (-157, 160), r"$\alpha_L$": (57, 39)}.items():
    ax1.plot(x, y, "rx", ms=10, mew=2)
    ax1.annotate(nome, (x, y), textcoords="offset points", xytext=(8, 6),
                 color="red", fontsize=10)

ax2.plot(traj.time / 1000, phi_deg, lw=0.8, label=r"$\phi$")
ax2.plot(traj.time / 1000, psi_deg, lw=0.8, label=r"$\psi$")
ax2.set(xlabel="tempo (ns)", ylabel="ângulo (graus)",
        ylim=(-180, 180), yticks=range(-180, 181, 90),
        title="Séries temporais")
ax2.legend(loc="upper right")

fig.tight_layout()
out = cfg.FIGS / "00_ramachandran_smoke.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
print(f"figura salva em {out}")

## 6. Disparar as réplicas de produção

Com o ciclo validado, as três réplicas de 500 ns do Sistema A rodam em background. Elas usam sementes distintas (`src/config.py::replica_seed`), o que as torna estatisticamente independentes — condição para o teste de convergência da Fase 1.

Executar **no PowerShell**, fora do notebook (a produção leva dias e não deve depender do kernel ficar vivo):

```powershell
.\scripts\run_replicas.ps1 -System A
```

Acompanhar o progresso:

```powershell
Get-Content .\logs\sysA-rep1-*.log -Tail 20 -Wait
```

As réplicas rodam em sequência porque compartilham a mesma GPU — em paralelo elas dividiriam a banda da placa sem reduzir o tempo total.

---

### Fase 0 concluída quando:

- [ ] `python -m openmm.testInstallation` lista CUDA
- [ ] `pytest -q` passa
- [ ] este notebook roda inteiro sem erro
- [ ] `figs/00_ramachandran_smoke.png` existe
- [ ] réplicas do Sistema A rodando em background

**Próximo passo — Fase 1:** `src/features.py` (diedros, raio de giro, RMSD) e `notebooks/01_features.ipynb`.